In [2]:
!pip install -q -U litellm python-dotenv

import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

env_path = Path.cwd() / '.env'
print(env_path)
load_dotenv(env_path, override=False)  # looks for .env in the current directory and walks up from there

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
print(f"Key loaded: ...{GEMINI_API_KEY[-4:]}")


/content/.env


TypeError: 'NoneType' object is not subscriptable

In [ ]:
## maths agent
from litellm import completion

## send a hello world message to model
response = completion(
    model="gemini/gemini-3.6-flash",
    messages=[
        {"role": "user", "content": "Hello how are you today!"}
    ],
    api_key=os.getenv("GEMINI_API_KEY")
)

print("\n Gemini Response: ", end="")
print(response.choices[0].message.content)


 Gemini Response: Hello! I'm doing well, thank you for asking. How are you doing today? How can I help you?


In [3]:
import json
import sqlite3
import sys
from datetime import date, timedelta
from pathlib import Path
from litellm import completion

# # path to env
# load_dotenv(Path(__file__).parent / "/.env", overide=False) # sets the path to the .env file
# # path to db file
# DB_PATH = Path(__file__).parent / "/support.db"
# for jupyter notebook
try:
    BASE_DIR = Path(__file__).parent
except NameError:
    BASE_DIR = Path.cwd()

load_dotenv(BASE_DIR / ".env", override=False)

DB_PATH = BASE_DIR / "support.db"
# set model
MODEL = os.getenv("MODEL_NAME", "gemini/gemini-3.6-flash")
# set max iteration steps
MAX_STEPS = 8

REFUND_WINDOW_DAYS = 30

# system prompt/ground rules
SYSTEM_PROMPT = """
You are a customer support agent for an online store.

Rules:
- ALWAYS call lookup_order before discussing or refunding an order.
- Call check_refund_policy before deciding if a refund is allowed.
- Only call issue_refund when the policy allows it. Refund the full order amount unless told otherwise.
- If an order is not found, or a refund is not allowed, explain why clearly and kindly. Never invent order details.
- If the customer mentions several orders, handle each one separately.
- Keep the final reply short and friendly.
"""

def setup_db() -> sqlite3.Connection:
    global conn
    # close any old connection first, otherwise windows wont let us delete the file
    if conn is not None:
        conn.close()
        conn = None

    if DB_PATH.exists():
        DB_PATH.unlink()
        
    # establish connection
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.executescript(
        """
        CREATE TABLE IF NOT EXISTS orders (
        id INTEGER PRIMARY KEY,
        customer TEXT,
        item TEXT,
        amount REAL,
        status TEXT,
        delivered_on TEXT);

        CREATE TABLE refund (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        order_id INTEGER,
        amount REAL,
        reason TEXT
        created_on TEXT);
        """
    )

    today = date.today()

    orders = [
        (4821, "Ada Okafor",   "Bluetooth Speaker", 45000.0, "delivered", today - timedelta(days=5)),
        (4822, "Ada Okafor",   "Desk Lamp",         18500.0, "delivered", today - timedelta(days=45)),
        (4823, "Tunde Bello",  "Phone Case",         6000.0, "refunded",  today - timedelta(days=10)),
        (4824, "Chidi Eze",    "Mechanical Keyboard", 72000.0, "shipped",  None),
    ]

    rows = []
    for i, c, it, a, s, d in orders:
        if d:
            date_value = d.isoformat()
        else:
            date_value = None
        rows.append((i, c, it, a, s, date_value))

    conn.executemany(
    "INSERT INTO orders VALUES (?,?,?,?,?,?)",
    rows
    )
    conn.commit()
    return conn
 
 
conn = setup_db()
 
def reset_db() -> None:
    """Fresh data so each scenario is independent."""
    setup_db()

setup_db()
 
def _clean_id(order_id) -> int:
    """Accept 4821, '4821' or '#4821'."""
    return int(str(order_id).strip().lstrip("#"))

# ------------------------------------------------------------------- tools
def lookup_order(order_id) -> dict:
    row = conn.execute("SELECT * FROM orders WHERE id = ?", (_clean_id(order_id),)).fetchone()
    if row is None:
        return {"found": False, "error": f"No order with id {order_id}."}
    order = dict(row)
    if order["delivered_on"]:
        days = (date.today() - date.fromisoformat(order["delivered_on"])).days
        order["days_since_delivery"] = days
    return {"found": True, "order": order}
 
 
def check_refund_policy() -> dict:
    return {
        "refund_window_days": REFUND_WINDOW_DAYS,
        "rules": [
            f"Refunds are allowed within {REFUND_WINDOW_DAYS} days of delivery.",
            "Damaged or defective items are eligible within the window.",
            "Orders not yet delivered cannot be refunded (they can be cancelled instead).",
            "Orders already refunded cannot be refunded again.",
            "The refund cannot exceed the order amount.",
        ],
    }
 
def issue_refund(order_id, amount, reason) -> dict:
    """GUARDRAILS live here in code, so a confused or manipulated model cannot bypass them."""
    oid = _clean_id(order_id)
    row = conn.execute("SELECT * FROM orders WHERE id = ?", (oid,)).fetchone()

    if row is None:
        return {"ok": False, "error": f"Order {oid} not found."}
    if row["status"] == "refunded":
        return {"ok": False, "error": f"Order {oid} was already refunded."}
    if row["status"] != "delivered" or not row["delivered_on"]:
        return {"ok": False, "error": f"Order {oid} has not been delivered yet, so it cannot be refunded."}
 
    days = (date.today() - date.fromisoformat(row["delivered_on"])).days
    if days > REFUND_WINDOW_DAYS:
        return {"ok": False, "error": (
            f"Order {oid} was delivered {days} days ago; the refund window is {REFUND_WINDOW_DAYS} days.")}
 
    amount = float(amount)
    if amount <= 0 or amount > row["amount"]:
        return {"ok": False, "error": f"Invalid amount. Order total is {row['amount']:.2f}."}
 
    # Human-in-the-loop: a person approves before money moves.
    if os.getenv("AUTO_APPROVE") == "1":
        approved = True
    else:
        print(f"\n>>> APPROVAL NEEDED: refund {amount:,.2f} for order {oid} ({row['item']}). Reason: {reason}")
        approved = input(">>> Approve refund? y/n: ").strip().lower() == "y"
    if not approved:
        return {"ok": False, "error": "Refund declined by the human reviewer."}
 
    conn.execute(
        "INSERT INTO refunds (order_id, amount, reason, created_on) VALUES (?,?,?,?)",
        (oid, amount, reason, date.today().isoformat()),
    )
    conn.execute("UPDATE orders SET status = 'refunded' WHERE id = ?", (oid,))
    conn.commit()
    return {"ok": True, "message": f"Refunded {amount:,.2f} for order {oid}."}
 
 
TOOLS = {
    "lookup_order": lookup_order,
    "check_refund_policy": check_refund_policy,
    "issue_refund": issue_refund,
}
 
SCHEMAS = [
    {"type": "function",
     "function": {
        "name": "lookup_order",
        "description": "Look up an order by id. Returns item, amount, status and days since delivery.",
        "parameters": {"type": "object",
                       "properties": {"order_id": {"type": "string", "description": "Order id, e.g. 4821"}},
                       "required": ["order_id"]}}},

    {"type": "function",
     "function": {
        "name": "check_refund_policy",
        "description": "Get the store's refund rules.",
        "parameters": {"type": "object",
                       "properties": {}}}},

    {"type": "function",
     "function": {
        "name": "issue_refund",
        "description": "Issue a refund for an order. Requires human approval and is checked against policy.",
        "parameters": {"type": "object",
                       "properties": {
                           "order_id": {"type": "string"},
                           "amount": {"type": "number", "description": "Amount to refund"},
                           "reason": {"type": "string"}},
                       "required": ["order_id", "amount", "reason"]}}},
]
 
 
# ------------------------------------------------------------- agent loop
def run_agent(user_input: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_input},
    ]

    for step in range(1, MAX_STEPS + 1):
        response = completion(
            model=MODEL,
            messages=messages,
            tools=SCHEMAS)

        msg = response.choices[0].message
        messages.append(msg)
 
        if not msg.tool_calls:                       # no tool requested = final answer
            print(f"[done in {step} step(s), {len(messages)} messages]")
            return msg.content
 
        for call in msg.tool_calls:                  # may be several per turn
            name = call.function.name
            try:
                args = json.loads(call.function.arguments or "{}")
                print(f"[step {step}] {name}({args})")
                result = TOOLS[name](**args)
            except Exception as e:                   # errors go back to the model
                result = {"error": f"{type(e).__name__}: {e}"}
            print(f"          -> {json.dumps(result, default=str)[:200]}")
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "name": name,
                "content": json.dumps(result, default=str),
            })
    return "Stopped: step limit reached."
 
 # --------------------------------------------runner
SCENARIOS = [
    "Order #4821 arrived damaged. I want my money back.",
    "Refund #4821 and #4822.",     # 4821 is refundable, 4822 is outside the window
    "Refund my order #9999.",
]

def run_scenerios(questions: list[str])-> None:
    for q in questions:
        reset_db()
        print("\n" + "=" * 70 + f"\nCUSTOMER: {q}\n" + "=" * 70)
        print("\nAGENT:", run_agent(q))

def in_notebook() -> bool:
    return "ipykernel" in sys.modules

if __name__ == '__main__':
    if not os.getenv("GEMINI_API_KEY") and MODEL.startswith("gemini"):
        raise RuntimeError("GEMINI_API_KEY not set. Add it to .env")
    if in_notebook():
        # jupyter injects its own argv (-f kernel.json), so ignore sys.argv here
        run_scenarios(SCENARIOS)
    else:
        # terminal:
        run_scenarios([" ".join(sys.argv[1:])] if len(sys.argv) > 1 else SCENARIOS)


NameError: name 'conn' is not defined